# 03 — AIA CNN Realistic Class-Imbalance Year-Holdout Sanity Experiment

**Purpose:** stress-test the AIA image CNN under a more realistic imbalanced flare-forecasting setup before moving into the formal literature-style baseline.

This notebook is **not a final publication result**. It is an engineering/scientific sanity experiment.

**Design**

- Dataset: baseline AIA 2010–2016 manifest
- Train years: 2010–2013
- Validation year: 2014
- Input: six-channel AIA tensor, originally `512 × 512 × 6`
- Model input resolution: resized to `224 × 224`
- Label: `label_48h_final`
- Important: ignore embedded NPZ `y`; use repaired manifest label only
- Class handling: class-weighted BCE loss using `pos_weight`
- Metrics: ROC-AUC, PR-AUC, precision, recall, specificity, F1, TSS, HSS
- Thresholds: report both fixed `0.5` and validation best-TSS threshold


In [ ]:
from pathlib import Path
import hashlib
import json
import subprocess
import random
import time
from copy import deepcopy

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch import nn
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import roc_auc_score, average_precision_score

SEED = 42

ROOT = Path.home() / "solar_flare_aia"
MANIFEST = ROOT / "training/final_metadata/baseline_2010_2016_AR_SPECIFIC_manifest.csv"

CACHE_DIR = ROOT / "cache/gcs_npz_realistic_imbalance"
METRICS_DIR = ROOT / "results/metrics"
MODELS_DIR = ROOT / "results/models"

CACHE_DIR.mkdir(parents=True, exist_ok=True)
METRICS_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

EXPERIMENT_NAME = "realistic_imbalance_aia_cnn_year_holdout_10to1"

TRAIN_YEARS = [2010, 2011, 2012, 2013]
VAL_YEARS = [2014]

# Cost-aware realistic-imbalance sanity setting.
# This is more realistic than balanced 1:1, but still smaller than the full 65k/2.4k dataset.
TRAIN_POS = 150
NEG_PER_POS = 10
TRAIN_NEG = TRAIN_POS * NEG_PER_POS

VAL_POS = 75
VAL_NEG = VAL_POS * NEG_PER_POS

IMAGE_SIZE = 224
BATCH_SIZE = 16
EPOCHS = 5
LR = 3e-4
WEIGHT_DECAY = 1e-4
NUM_WORKERS = 0

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM GB:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
else:
    print("WARNING: CUDA is not available. Do not run training until GPU is available.")

In [ ]:
df = pd.read_csv(MANIFEST, low_memory=False)

print("Manifest:", MANIFEST)
print("Rows:", len(df))
print("\nColumns:")
print(df.columns.tolist())

print("\nYear counts:")
print(df["year"].value_counts().sort_index())

print("\nOverall labels:")
print(df["label_48h_final"].value_counts())

print("\nLabels by year:")
display(pd.crosstab(df["year"], df["label_48h_final"]))

print("\nMissing checks:")
print("Missing gcp_path:", df["gcp_path"].isna().sum())
print("Missing sample_id:", df["sample_id"].isna().sum())
print("Missing label_48h_final:", df["label_48h_final"].isna().sum())

In [ ]:
def sample_imbalanced_subset(df, years, n_pos, n_neg, seed):
    part = df[df["year"].isin(years)].copy()

    pos_pool = part[part["label_48h_final"] == 1]
    neg_pool = part[part["label_48h_final"] == 0]

    actual_pos = min(n_pos, len(pos_pool))
    actual_neg = min(n_neg, len(neg_pool))

    if actual_pos < n_pos:
        print(f"WARNING: requested {n_pos} positives but only {actual_pos} available for years={years}")
    if actual_neg < n_neg:
        print(f"WARNING: requested {n_neg} negatives but only {actual_neg} available for years={years}")

    pos = pos_pool.sample(actual_pos, random_state=seed)
    neg = neg_pool.sample(actual_neg, random_state=seed)

    out = pd.concat([pos, neg]).sample(frac=1, random_state=seed).reset_index(drop=True)
    return out

train_df = sample_imbalanced_subset(
    df, TRAIN_YEARS, TRAIN_POS, TRAIN_NEG, SEED
)

val_df = sample_imbalanced_subset(
    df, VAL_YEARS, VAL_POS, VAL_NEG, SEED + 1
)

print("Train subset rows:", len(train_df))
print(train_df["label_48h_final"].value_counts())
print("\nTrain years:")
print(pd.crosstab(train_df["year"], train_df["label_48h_final"]))

print("\nValidation subset rows:", len(val_df))
print(val_df["label_48h_final"].value_counts())
print("\nValidation years:")
print(pd.crosstab(val_df["year"], val_df["label_48h_final"]))

train_samples_path = METRICS_DIR / f"{EXPERIMENT_NAME}_train_samples.csv"
val_samples_path = METRICS_DIR / f"{EXPERIMENT_NAME}_val_samples.csv"

train_df.to_csv(train_samples_path, index=False)
val_df.to_csv(val_samples_path, index=False)

print("\nSaved train samples:", train_samples_path)
print("Saved val samples:", val_samples_path)

In [ ]:
def local_cache_path(gcp_path: str) -> Path:
    safe = hashlib.md5(gcp_path.encode()).hexdigest() + ".npz"
    return CACHE_DIR / safe

def cache_gcs_file(gcp_path: str) -> Path:
    local_path = local_cache_path(gcp_path)
    if not local_path.exists():
        subprocess.run(["gcloud", "storage", "cp", gcp_path, str(local_path)], check=True)
    return local_path

def precache_frame(frame: pd.DataFrame, name: str):
    paths = frame["gcp_path"].tolist()
    total = len(paths)
    start = time.time()

    print(f"Pre-caching {name}: {total} files")
    for i, gcp_path in enumerate(paths, 1):
        local_path = local_cache_path(gcp_path)
        if not local_path.exists():
            print(f"[{name}] downloading {i}/{total}: {gcp_path}")
            cache_gcs_file(gcp_path)

        if i % 100 == 0 or i == total:
            elapsed = time.time() - start
            print(f"[{name}] cached/checked {i}/{total} files | elapsed {elapsed/60:.1f} min")

    print(f"Finished pre-caching {name}")

# Run this cell before training. It may take time the first time, but it prevents downloads during batches.
precache_frame(train_df, "train")
precache_frame(val_df, "val")

In [ ]:
class AIANPZDataset(Dataset):
    def __init__(self, frame: pd.DataFrame, image_size: int = 224):
        self.frame = frame.reset_index(drop=True)
        self.image_size = image_size

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, idx):
        row = self.frame.iloc[idx]
        local_path = cache_gcs_file(row["gcp_path"])

        data = np.load(local_path, allow_pickle=True)

        # Official image tensor is H, W, C = 512, 512, 6
        x = data["x"].astype(np.float32)
        x = np.nan_to_num(x, nan=0.0, posinf=1.0, neginf=0.0)
        x = np.clip(x, 0.0, 1.0)

        # Convert to PyTorch C, H, W = 6, 512, 512
        x = torch.from_numpy(np.transpose(x, (2, 0, 1))).float()

        # Resize to reduce compute cost
        if self.image_size != 512:
            x = F.interpolate(
                x.unsqueeze(0),
                size=(self.image_size, self.image_size),
                mode="bilinear",
                align_corners=False,
            ).squeeze(0)

        # Important: use repaired manifest label, not embedded NPZ y
        y = torch.tensor(float(row["label_48h_final"]), dtype=torch.float32)

        return x, y, str(row["sample_id"])

train_loader = DataLoader(
    AIANPZDataset(train_df, image_size=IMAGE_SIZE),
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)

val_loader = DataLoader(
    AIANPZDataset(val_df, image_size=IMAGE_SIZE),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)

x_batch, y_batch, sample_ids = next(iter(train_loader))
print("x batch shape:", tuple(x_batch.shape))
print("y batch shape:", tuple(y_batch.shape))
print("y positives in batch:", int(y_batch.sum().item()))
print("x min/max:", float(x_batch.min()), float(x_batch.max()))
print("sample ids:", list(sample_ids[:3]))

In [ ]:
class SmallAIAImbalanceCNN(nn.Module):
    def __init__(self, in_channels=6):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(in_channels, 32, kernel_size=7, stride=2, padding=3),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(128, 192, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(192),
            nn.ReLU(inplace=True),

            nn.AdaptiveAvgPool2d((1, 1)),
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.25),
            nn.Linear(192, 64),
            nn.ReLU(inplace=True),
            nn.Dropout(0.25),
            nn.Linear(64, 1),
        )

    def forward(self, x):
        return self.classifier(self.features(x)).squeeze(1)

model = SmallAIAImbalanceCNN(in_channels=6).to(device)

with torch.no_grad():
    test_logits = model(x_batch.to(device))

print(model.__class__.__name__)
print("Test logits shape:", tuple(test_logits.shape))
print("Model device:", next(model.parameters()).device)
print("Trainable parameters:", sum(p.numel() for p in model.parameters() if p.requires_grad))

In [ ]:
def safe_auc(y_true, y_prob):
    try:
        return float(roc_auc_score(y_true, y_prob))
    except ValueError:
        return None

def safe_ap(y_true, y_prob):
    try:
        return float(average_precision_score(y_true, y_prob))
    except ValueError:
        return None

def binary_metrics_at_threshold(y_true, y_prob, threshold=0.5):
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob).astype(float)
    y_pred = (y_prob >= threshold).astype(int)

    tp = int(((y_true == 1) & (y_pred == 1)).sum())
    tn = int(((y_true == 0) & (y_pred == 0)).sum())
    fp = int(((y_true == 0) & (y_pred == 1)).sum())
    fn = int(((y_true == 1) & (y_pred == 0)).sum())

    eps = 1e-12

    accuracy = (tp + tn) / max(tp + tn + fp + fn, 1)
    precision = tp / max(tp + fp, 1)
    recall = tp / max(tp + fn, 1)
    specificity = tn / max(tn + fp, 1)
    f1 = 2 * precision * recall / max(precision + recall, eps)
    tss = recall + specificity - 1

    numerator = 2 * (tp * tn - fp * fn)
    denominator = ((tp + fn) * (fn + tn) + (tp + fp) * (fp + tn))
    hss = numerator / max(denominator, eps)

    return {
        "threshold": float(threshold),
        "accuracy": float(accuracy),
        "precision": float(precision),
        "recall": float(recall),
        "specificity": float(specificity),
        "f1": float(f1),
        "tss": float(tss),
        "hss": float(hss),
        "tp": tp,
        "tn": tn,
        "fp": fp,
        "fn": fn,
    }

def best_tss_threshold(y_true, y_prob):
    y_prob = np.asarray(y_prob).astype(float)
    thresholds = np.unique(np.round(y_prob, 6))
    if len(thresholds) > 400:
        thresholds = np.linspace(0.0, 1.0, 401)

    best = None
    for thr in thresholds:
        m = binary_metrics_at_threshold(y_true, y_prob, threshold=float(thr))
        if best is None or m["tss"] > best["tss"]:
            best = m

    return best

def evaluate_predictions(y_true, y_prob):
    fixed = binary_metrics_at_threshold(y_true, y_prob, threshold=0.5)
    best = best_tss_threshold(y_true, y_prob)

    return {
        "roc_auc": safe_auc(y_true, y_prob),
        "pr_auc": safe_ap(y_true, y_prob),
        "metrics_at_0_5": fixed,
        "metrics_at_best_tss_threshold": best,
    }

def collect_predictions(model, loader, device):
    model.eval()
    rows = []
    y_true, y_prob = [], []

    with torch.no_grad():
        for x, y, sample_ids in loader:
            x = x.to(device, non_blocking=True)
            logits = model(x)
            prob = torch.sigmoid(logits).detach().cpu().numpy()

            y_np = y.numpy()
            for sid, yy, pp in zip(sample_ids, y_np, prob):
                rows.append({
                    "sample_id": sid,
                    "y_true": int(yy),
                    "y_prob": float(pp),
                })

            y_true.extend(y_np.tolist())
            y_prob.extend(prob.tolist())

    pred_df = pd.DataFrame(rows)
    metrics = evaluate_predictions(y_true, y_prob)
    return pred_df, metrics

In [ ]:
pos_count = int(train_df["label_48h_final"].sum())
neg_count = int(len(train_df) - pos_count)
pos_weight_value = neg_count / max(pos_count, 1)

print("Train positive count:", pos_count)
print("Train negative count:", neg_count)
print("pos_weight:", pos_weight_value)

model = SmallAIAImbalanceCNN(in_channels=6).to(device)
criterion = nn.BCEWithLogitsLoss(
    pos_weight=torch.tensor([pos_weight_value], dtype=torch.float32, device=device)
)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

history = []
best_record = None
best_state = None

for epoch in range(1, EPOCHS + 1):
    epoch_start = time.time()
    model.train()
    losses = []

    for step, (x, y, sample_ids) in enumerate(train_loader, 1):
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()

        losses.append(float(loss.item()))

        if step == 1:
            print(f"Epoch {epoch} first batch x:", tuple(x.shape), "y positives:", int(y.sum().item()))

    train_pred_df, train_eval = collect_predictions(model, train_loader, device)
    val_pred_df, val_eval = collect_predictions(model, val_loader, device)

    record = {
        "epoch": epoch,
        "train_loss": float(np.mean(losses)),
        "train_eval": train_eval,
        "val_eval": val_eval,
        "elapsed_minutes": float((time.time() - epoch_start) / 60.0),
    }
    history.append(record)

    val_best_tss = val_eval["metrics_at_best_tss_threshold"]["tss"]
    if best_record is None or val_best_tss > best_record["val_eval"]["metrics_at_best_tss_threshold"]["tss"]:
        best_record = deepcopy(record)
        best_state = deepcopy(model.state_dict())

    print(f"\nEpoch {epoch}/{EPOCHS}")
    print("Train loss:", record["train_loss"])
    print("Train PR-AUC:", train_eval["pr_auc"], "ROC-AUC:", train_eval["roc_auc"])
    print("Val PR-AUC:", val_eval["pr_auc"], "ROC-AUC:", val_eval["roc_auc"])
    print("Val @0.5:", val_eval["metrics_at_0_5"])
    print("Val @best TSS:", val_eval["metrics_at_best_tss_threshold"])
    print("Elapsed min:", round(record["elapsed_minutes"], 2))

print("\nBest epoch by validation TSS:", best_record["epoch"])
print("Best validation metrics:", best_record["val_eval"]["metrics_at_best_tss_threshold"])

In [ ]:
# Save outputs
if best_state is not None:
    model.load_state_dict(best_state)

train_pred_df, train_eval_final = collect_predictions(model, train_loader, device)
val_pred_df, val_eval_final = collect_predictions(model, val_loader, device)

train_pred_path = METRICS_DIR / f"{EXPERIMENT_NAME}_train_predictions.csv"
val_pred_path = METRICS_DIR / f"{EXPERIMENT_NAME}_val_predictions.csv"
metrics_path = METRICS_DIR / f"{EXPERIMENT_NAME}_metrics.json"
summary_path = METRICS_DIR / f"{EXPERIMENT_NAME}_readable_summary.md"
model_path = MODELS_DIR / f"{EXPERIMENT_NAME}.pt"

train_pred_df.to_csv(train_pred_path, index=False)
val_pred_df.to_csv(val_pred_path, index=False)

output = {
    "purpose": "Realistic class-imbalance AIA CNN year-holdout sanity experiment; not final publication result",
    "experiment_name": EXPERIMENT_NAME,
    "manifest": str(MANIFEST),
    "uses_label": "label_48h_final",
    "ignores_npz_y": True,
    "train_years": TRAIN_YEARS,
    "val_years": VAL_YEARS,
    "train_rows": int(len(train_df)),
    "val_rows": int(len(val_df)),
    "train_label_counts": {str(k): int(v) for k, v in train_df["label_48h_final"].value_counts().to_dict().items()},
    "val_label_counts": {str(k): int(v) for k, v in val_df["label_48h_final"].value_counts().to_dict().items()},
    "requested_train_pos": TRAIN_POS,
    "requested_train_neg": TRAIN_NEG,
    "requested_val_pos": VAL_POS,
    "requested_val_neg": VAL_NEG,
    "image_size": IMAGE_SIZE,
    "original_input_shape_hwc": [512, 512, 6],
    "model_input_shape_chw": [6, IMAGE_SIZE, IMAGE_SIZE],
    "batch_size": BATCH_SIZE,
    "epochs": EPOCHS,
    "lr": LR,
    "weight_decay": WEIGHT_DECAY,
    "pos_weight": float(pos_weight_value),
    "model_name": model.__class__.__name__,
    "best_epoch_by_val_tss": int(best_record["epoch"]),
    "history": history,
    "final_train_eval": train_eval_final,
    "final_val_eval": val_eval_final,
    "model_path": str(model_path),
    "train_predictions_path": str(train_pred_path),
    "val_predictions_path": str(val_pred_path),
}

metrics_path.write_text(json.dumps(output, indent=2))
torch.save(model.state_dict(), model_path)

best_val = val_eval_final["metrics_at_best_tss_threshold"]
fixed_val = val_eval_final["metrics_at_0_5"]

summary_text = f'''# Realistic Class-Imbalance AIA CNN Year-Holdout Sanity Experiment

Purpose: {output["purpose"]}

Experiment: `{EXPERIMENT_NAME}`

## Protocol

- Train years: {TRAIN_YEARS}
- Validation years: {VAL_YEARS}
- Train rows: {len(train_df)}
- Validation rows: {len(val_df)}
- Train label counts: {output["train_label_counts"]}
- Validation label counts: {output["val_label_counts"]}
- Original input: 512 × 512 × 6
- Model input: {IMAGE_SIZE} × {IMAGE_SIZE} × 6
- Label used: `label_48h_final`
- Embedded NPZ `y`: ignored
- Loss: weighted BCEWithLogitsLoss
- pos_weight: {pos_weight_value:.4f}

## Final validation metrics at fixed threshold 0.5

- ROC-AUC: {val_eval_final["roc_auc"]}
- PR-AUC: {val_eval_final["pr_auc"]}
- Accuracy: {fixed_val["accuracy"]:.4f}
- Precision: {fixed_val["precision"]:.4f}
- Recall: {fixed_val["recall"]:.4f}
- Specificity: {fixed_val["specificity"]:.4f}
- F1: {fixed_val["f1"]:.4f}
- TSS: {fixed_val["tss"]:.4f}
- HSS: {fixed_val["hss"]:.4f}
- TP: {fixed_val["tp"]}
- TN: {fixed_val["tn"]}
- FP: {fixed_val["fp"]}
- FN: {fixed_val["fn"]}

## Final validation metrics at best validation TSS threshold

- Threshold: {best_val["threshold"]:.6f}
- Accuracy: {best_val["accuracy"]:.4f}
- Precision: {best_val["precision"]:.4f}
- Recall: {best_val["recall"]:.4f}
- Specificity: {best_val["specificity"]:.4f}
- F1: {best_val["f1"]:.4f}
- TSS: {best_val["tss"]:.4f}
- HSS: {best_val["hss"]:.4f}
- TP: {best_val["tp"]}
- TN: {best_val["tn"]}
- FP: {best_val["fp"]}
- FN: {best_val["fn"]}

## Important note

This is a sanity experiment under controlled imbalanced sampling, not a final publication result.
'''

summary_path.write_text(summary_text)

print("Saved metrics:", metrics_path)
print("Saved readable summary:", summary_path)
print("Saved train predictions:", train_pred_path)
print("Saved val predictions:", val_pred_path)
print("Saved model:", model_path)

In [ ]:
from IPython.display import display, Markdown

rows = []
for item in history:
    val_fixed = item["val_eval"]["metrics_at_0_5"]
    val_best = item["val_eval"]["metrics_at_best_tss_threshold"]

    rows.append({
        "epoch": item["epoch"],
        "train_loss": item["train_loss"],
        "val_roc_auc": item["val_eval"]["roc_auc"],
        "val_pr_auc": item["val_eval"]["pr_auc"],
        "val_tss_at_0_5": val_fixed["tss"],
        "val_f1_at_0_5": val_fixed["f1"],
        "best_tss_threshold": val_best["threshold"],
        "val_best_tss": val_best["tss"],
        "val_best_hss": val_best["hss"],
        "val_best_precision": val_best["precision"],
        "val_best_recall": val_best["recall"],
        "val_best_specificity": val_best["specificity"],
        "val_best_tp": val_best["tp"],
        "val_best_tn": val_best["tn"],
        "val_best_fp": val_best["fp"],
        "val_best_fn": val_best["fn"],
    })

summary_df = pd.DataFrame(rows)

display(Markdown("## Realistic class-imbalance sanity experiment — epoch summary"))
display(summary_df)

display(Markdown("## Final validation result at best-TSS threshold"))
display(pd.DataFrame([val_eval_final["metrics_at_best_tss_threshold"]]))

display(Markdown(f"""
**Final ROC-AUC:** {val_eval_final["roc_auc"]}

**Final PR-AUC:** {val_eval_final["pr_auc"]}

**Important:** This is not a final publication result. It is a realistic-imbalance sanity test before the formal AIA baseline.
"""))

## After the notebook finishes

Run these commands in the **VS Code terminal** to back up and commit outputs.

```bash
cd ~/solar_flare_aia

gcloud storage cp notebooks/training/03_aia_cnn_realistic_imbalance_year_holdout.ipynb \
  gs://suryabench-sharp-pipeline-bamidele/training_docs/

gcloud storage cp results/metrics/realistic_imbalance_aia_cnn_year_holdout_10to1_* \
  gs://suryabench-sharp-pipeline-bamidele/training_docs/

gcloud storage cp results/models/realistic_imbalance_aia_cnn_year_holdout_10to1.pt \
  gs://suryabench-sharp-pipeline-bamidele/training_models/

git add notebooks/training/03_aia_cnn_realistic_imbalance_year_holdout.ipynb
git add results/metrics/realistic_imbalance_aia_cnn_year_holdout_10to1_*

git commit -m "Add realistic-imbalance AIA CNN year-holdout experiment"
git push
```

Then stop the VM from your Mac Terminal or VS Code terminal:

```bash
gcloud compute instances stop solar-flare-aia-training-l4-c \
  --project=sonorous-shore-450510-i4 \
  --zone=europe-west4-c
```
